# Traffic Data Visualization
A step-by-step walkthrough to call the FastAPI traffic service, transform its response, and render road segments on an interactive Mapbox map.

## 1. Setup

In [36]:
# Install dependencies if needed
# !pip install requests pandas geopandas mapboxgl shapely

In [ ]:
import requests
import pandas as pd
from shapely import wkt
from shapely.geometry import mapping
from mapboxgl.viz import ChoroplethViz
from mapboxgl.utils import create_color_stops

MAPBOX_TOKEN = ""
BASE_URL = "http://localhost:8000"

# Jacksonville, FL center coordinates
MAP_CENTER = [-81.6557, 30.3322]

## 2. API Request
Fetch aggregated speed data for Monday AM Peak.

In [38]:
params = {"day": "Monday", "period": "AM Peak", "limit": 10}
response = requests.get(f"{BASE_URL}/aggregates/", params=params)
response.raise_for_status()

records = response.json()
print(f"Fetched {len(records)} links from the API.")
records[:2]  # preview first 2 records

Fetched 10 links from the API.


[{'link_id': 16981048,
  'road_name': 'Philips Hwy',
  'length': 0.009320565,
  'geometry_wkt': 'LINESTRING(-81.59791 30.24124,-81.59801 30.24135)',
  'average_speed': 45.40133333333333},
 {'link_id': 16981074,
  'road_name': 'Philips Hwy',
  'length': 0.008699194,
  'geometry_wkt': 'LINESTRING(-81.59775 30.24135,-81.59784 30.24146)',
  'average_speed': 49.399333333333324}]

## 3. Data Transformation
The API returns `geometry_wkt` as a `LINESTRING(...)` string. We need to convert it to GeoJSON `Feature` objects that Mapbox can render.

> **Note:** WKT (Well-Known Text) is a compact PostGIS text format. `shapely.wkt.loads()` parses it, and `shapely.geometry.mapping()` converts it to a GeoJSON-compatible dict.

In [39]:
def to_feature(record):
    """Convert a single API record to a GeoJSON Feature."""
    geometry = mapping(wkt.loads(record["geometry_wkt"]))
    return {
        "type": "Feature",
        "geometry": geometry,
        "properties": {
            "link_id": record["link_id"],
            "road_name": record.get("road_name"),
            "average_speed": round(record["average_speed"], 1),
        },
    }

# Filter out records without geometry and build the FeatureCollection
features = [to_feature(r) for r in records if r.get("geometry_wkt")]
if not features:
    print("No data available for selected filters")
feature_collection = {
    "type": "FeatureCollection",
    "features": features
}

feature_collection = {"type": "FeatureCollection", "features": features}
print(f"{len(features)} features ready for visualization.")

10 features ready for visualization.


## 4. Visualization
Color road segments by `average_speed` using a choropleth style.
- **Red / Orange** = Slow (congested)
- **Green / Blue** = Fast (free-flowing)

In [40]:
# Define speed breakpoints for the color ramp
speed_stops = create_color_stops([5, 15, 25, 35, 45, 55], colors="RdYlGn")

viz = ChoroplethViz(
    feature_collection,
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=speed_stops,
    color_default="#888888",
    line_width=2,
    opacity=0.85,
    center=MAP_CENTER,
    zoom=11,
)

viz.show()

## 5. Summary Table
The 10 slowest road segments during Monday AM Peak.

In [41]:
df = pd.DataFrame([
    {
        "link_id": r["link_id"],
        "avg_speed (mph)": round(r["average_speed"], 1),
    }
    for r in records
])

df.sort_values("avg_speed (mph)").head(10).reset_index(drop=True)

,link_id,avg_speed (mph)
0,16981075,8.7
1,16993102,9.0
2,16993105,15.8
3,16981048,45.4
4,16981074,49.4
5,16993999,58.5
6,16993997,63.6
7,16993582,69.5
8,16993624,72.3
9,16993580,72.7


## Bonus: Spatial Filter
Instead of the entire city, narrow down to a specific bounding box of interest.

In [42]:
payload = {
    "min_lon": -81.7,
    "min_lat": 30.25,
    "max_lon": -81.6,
    "max_lat": 30.38,
    "day": "Monday",
    "period": "AM Peak",
}

response = requests.post(f"{BASE_URL}/aggregates/spatial_filter", json=payload)
response.raise_for_status()
spatial_records = response.json()
print(f"Found {len(spatial_records)} links inside bounding box.")

# Build a new GeoJSON for each link — speeds are nested within each link here
spatial_features = []
for link in spatial_records:
    if not link.get("geometry_wkt") or not link.get("speeds"):
        continue
    avg = sum(s["speed"] for s in link["speeds"]) / len(link["speeds"])
    spatial_features.append({
        "type": "Feature",
        "geometry": mapping(wkt.loads(link["geometry_wkt"])),
        "properties": {
            "link_id": link["id"],
            "average_speed": round(avg, 1),
            "road_name": link.get("road_name") or "Unknown",
        },
    })

spatial_viz = ChoroplethViz(
    {"type": "FeatureCollection", "features": spatial_features},
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=speed_stops,
    line_width=3,
    opacity=0.9,
    center=[-81.65, 30.31],
    zoom=13,
)

spatial_viz.show()

Found 17659 links inside bounding box.


/Users/andrenuneszardo/urbansdk-homework/.venv/lib/python3.14/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
